# Ranking

In this notebook, we will learn about the features of **PyBroker** that enable you to rank ticker symbols in your trading strategy. With these features, you can easily optimize your strategy and manage risk more effectively.

In [1]:
import pybroker
from pybroker import Strategy, StrategyConfig, YFinance

pybroker.enable_data_source_cache("ranking")

## Scoring Ticker Symbols

In this section, we will learn about how to rank ticker symbols when placing buy orders. Let's begin with an example of how to rank ticker symbols based on volume when placing buy orders. 

In [2]:
def buy_highest_volume(ctx):
    # If there are no long positions across all tickers being traded:
    if not tuple(ctx.long_positions()):
        ctx.buy_shares = ctx.calc_target_shares(1)
        ctx.hold_bars = 2
        ctx.long_score = ctx.volume[-1]

The ```buy_highest_volume``` function ranks ticker symbols by their most recent trading volume and allocates 100% of the portfolio for 2 bars. The ```ctx.score``` is set to ```ctx.volume[-1]```, which is the most recent trading volume.

In [3]:
config = StrategyConfig(max_long_positions=1)
strategy = Strategy(YFinance(), "6/1/2021", "6/1/2022", config)
strategy.add_execution(buy_highest_volume, ["T", "F", "GM", "PFE"])

To limit the number of long positions that can be held at any time to ```1```, we set [max_long_positions](https://www.pybroker.com/en/latest/reference/pybroker.config.html#pybroker.config.StrategyConfig.max_long_positions) to ```1``` in the [StrategyConfig](https://www.pybroker.com/en/latest/reference/pybroker.config.html#pybroker.config.StrategyConfig). In this example, we add the ```buy_highest_volume``` function to the [Strategy](https://www.pybroker.com/en/latest/reference/pybroker.strategy.html#pybroker.strategy.Strategy) object and specify the ticker symbols to trade: ```['T', 'F', 'GM', 'PFE']```.

In [4]:
result = strategy.backtest()
result.trades

Backtesting: 2021-06-01 00:00:00 to 2022-06-01 00:00:00

Loading bar data...


[*********************100%***********************]  4 of 4 completed


Loaded bar data: 0:00:00 

Test split: 2021-06-01 00:00:00 to 2022-05-31 00:00:00


100% (253 of 253) |######################| Elapsed Time: 0:00:00 Time:  0:00:00



Finished backtest: 0:00:01


,type,symbol,entry_date,exit_date,entry,exit,shares,pnl,return_pct,agg_pnl,bars,pnl_per_bar,stop,mae,mfe
id,,,,,,,,,,,,,,,
1,long,F,2021-06-02,2021-06-04,14.85,16.13,6734,8619.52,8.62,8619.52,2,4309.76,bar,-0.17,1.28
2,long,F,2021-06-07,2021-06-09,15.93,15.51,6801,-2856.42,-2.64,5763.10,2,-1428.21,bar,-0.60,0.27
3,long,F,2021-06-10,2021-06-14,15.43,15.06,6832,-2527.84,-2.40,3235.26,2,-1263.92,bar,-0.37,0.35
4,long,F,2021-06-15,2021-06-17,14.96,14.99,6900,207.00,0.20,3442.26,2,103.50,bar,-0.20,0.33
5,long,F,2021-06-18,2021-06-22,14.61,14.96,7003,2451.05,2.40,5893.31,2,1225.53,bar,-0.17,0.35
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80,long,F,2022-05-10,2022-05-12,13.43,12.47,7258,-6967.68,-7.15,-9482.76,2,-3483.84,bar,-0.96,0.41
81,long,F,2022-05-13,2022-05-17,13.25,13.34,6831,614.79,0.68,-8867.97,2,307.40,bar,-0.38,0.38
82,long,F,2022-05-18,2022-05-20,13.03,12.59,6735,-2963.40,-3.38,-11831.37,2,-1481.70,bar,-0.44,0.33


## Shorting the Lowest Scores

**PyBroker** can also rank short orders using [short_score](https://www.pybroker.com/en/latest/reference/pybroker.context.html#pybroker.context.ExecContext.short_score), where orders are placed for the ticker symbols with the *lowest* values. The following example buys the ticker symbol with the highest 5-day rate of change (ROC) while shorting the ticker symbol with the lowest 5-day ROC:

In [5]:
def long_high_short_low(ctx):
    # Wait for 6 bars of data and skip symbols with an open position:
    if ctx.bars < 6 or ctx.long_pos() or ctx.short_pos():
        return
    # Calculate the 5-day rate of change (ROC):
    roc = (ctx.close[-1] - ctx.close[-6]) / ctx.close[-6]
    if roc > 0 and not tuple(ctx.long_positions()):
        ctx.buy_shares = ctx.calc_target_shares(0.5)
        ctx.hold_bars = 2
        ctx.long_score = roc
    elif roc < 0 and not tuple(ctx.short_positions()):
        ctx.sell_shares = ctx.calc_target_shares(0.5)
        ctx.hold_bars = 2
        ctx.short_score = roc


strategy = Strategy(YFinance(), "1/1/2025", "1/1/2026")
strategy.add_execution(long_high_short_low, ["T", "F", "GM", "PFE"])
strategy.set_max_long_positions(1)
strategy.set_max_short_positions(1)
result = strategy.backtest()
result.trades

Backtesting: 2021-06-01 00:00:00 to 2022-06-01 00:00:00

Loaded cached bar data.

Test split: 2021-06-01 00:00:00 to 2022-05-31 00:00:00


100% (253 of 253) |######################| Elapsed Time: 0:00:00 Time:  0:00:00



Finished backtest: 0:00:00


,type,symbol,entry_date,exit_date,entry,exit,shares,pnl,return_pct,agg_pnl,bars,pnl_per_bar,stop,mae,mfe
id,,,,,,,,,,,,,,,
1,long,GM,2021-06-09,2021-06-11,63.43,61.58,782,-1446.70,-2.92,-1446.70,2,-723.35,bar,-2.26,0.73
2,short,T,2021-06-09,2021-06-11,21.93,22.08,2284,-342.60,-0.68,-1789.30,2,-171.30,bar,-0.19,0.12
3,long,PFE,2021-06-14,2021-06-16,39.73,39.56,1223,-207.91,-0.43,-1997.21,2,-103.96,bar,-0.35,0.34
4,short,F,2021-06-14,2021-06-16,15.06,15.09,3213,-96.39,-0.20,-2093.60,2,-48.20,bar,-0.24,0.30
5,long,T,2021-06-17,2021-06-21,21.85,21.76,2230,-200.70,-0.41,-2294.30,2,-100.35,bar,-0.39,0.17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
154,long,T,2022-05-18,2022-05-20,20.38,20.26,2151,-258.12,-0.59,-8297.79,2,-129.06,bar,-0.47,0.25
155,long,PFE,2022-05-23,2022-05-25,53.09,53.55,848,390.08,0.87,-7907.71,2,195.04,bar,-0.61,0.61
156,short,F,2022-05-23,2022-05-25,12.72,12.57,3668,550.20,1.19,-7357.51,2,275.10,bar,-0.23,0.45
